In [1]:
# ============================================================================
# BLOCK 1: INSTALL DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

!pip install ultralytics gradio pillow opencv-python-headless ipywidgets matplotlib -q

import torch
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import os
from ultralytics import YOLO

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print("="*70)

INSTALLING DEPENDENCIES



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ PyTorch: 2.6.0+cu118
✅ CUDA: True
✅ GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# ============================================================================
# BLOCK 2: LOAD YOLOv8 MODEL
# ============================================================================

print("="*70)
print("LOADING YOLOv8 SHOULDER/ARM MODEL")
print("="*70)

# Path to your best YOLOv8 model
MODEL_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt"

# Load model
model = YOLO(MODEL_PATH)

# Set to evaluation mode
model.model.eval()

print(f"✅ Model loaded: {MODEL_PATH}")
print(f"✅ Model type: YOLOv8m")
print(f"✅ Classes: {model.names}")
print("="*70)

LOADING YOLOv8 SHOULDER/ARM MODEL
✅ Model loaded: C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt
✅ Model type: YOLOv8m
✅ Classes: {0: 'fracture'}


In [5]:
# ============================================================================
# BLOCK 3: PREDICTION FUNCTION FOR YOLOv8
# ============================================================================

print("="*70)
print("DEFINING PREDICTION FUNCTION")
print("="*70)

def predict_fracture(image, threshold=0.3):
    """
    Detect fractures in X-ray image using YOLOv8
    Returns: (annotated_image, html_report, has_fracture, num_fractures)
    """
    # Convert to numpy array if PIL
    if isinstance(image, Image.Image):
        image = np.array(image)
    
    # Ensure RGB format
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    
    # Run YOLOv8 prediction
    results = model(image, conf=threshold, iou=0.5, verbose=False)
    
    # Get detection info
    boxes = results[0].boxes
    num_detections = len(boxes) if boxes else 0
    has_fracture = num_detections > 0
    
    if has_fracture:
        confidences = boxes.conf.cpu().numpy()
        max_confidence = float(np.max(confidences))
        avg_confidence = float(np.mean(confidences))
    else:
        max_confidence = 0.0
        avg_confidence = 0.0
    
    # Get annotated image
    annotated_img = results[0].plot()
    
    # Generate HTML report
    if has_fracture:
        result_html = f"""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; color: white;">
            <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
                🚨 FRACTURE DETECTED
            </h1>
        </div>

        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                📊 Detection Details
            </h2>

            <div style="background: #f7fafc; padding: 15px; border-radius: 8px; margin: 15px 0;">
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Number of fractures: <span style="font-weight: bold; color: #e53e3e;">{num_detections}</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Highest confidence: <span style="font-weight: bold; color: #38a169;">{max_confidence*100:.1f}%</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Average confidence: <span style="font-weight: bold;">{avg_confidence*100:.1f}%</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #667eea;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                🏥 Recommendation
            </h2>

            <div style="background: #fff5f5; border-left: 5px solid #e53e3e; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 16px; font-weight: bold; color: #c53030;">
                    IMMEDIATE MEDICAL CONSULTATION RECOMMENDED
                </p>
                <p style="margin: 10px 0 0 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    Fracture(s) detected in X-ray image.<br>
                    Please consult a qualified radiologist or orthopedic specialist.
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #667eea; padding-bottom: 10px;">
                ⚠️ Important Disclaimer
            </h2>

            <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
                <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
                    This is an AI-assisted screening tool for preliminary assessment.<br>
                    Results <strong>MUST</strong> be verified by a qualified medical professional.<br>
                    Do NOT use as sole basis for diagnosis or treatment decisions.
                </p>
            </div>
        </div>
        """
    else:
        result_html = f"""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); border-radius: 15px; color: white;">
            <h1 style="margin: 0; font-size: 32px; font-weight: bold; text-align: center;">
                ✅ NO FRACTURE DETECTED
            </h1>
        </div>

        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 25px; background: white; border-radius: 10px; margin-top: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 0; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                📊 Analysis Details
            </h2>

            <div style="background: #f0fff4; padding: 15px; border-radius: 8px; margin: 15px 0;">
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> No fractures detected
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> Threshold used: <span style="font-weight: bold;">{threshold}</span>
                </p>
                <p style="margin: 8px 0; font-size: 16px; color: #2d3748;">
                    <strong style="color: #48bb78;">•</strong> Image analyzed successfully
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                💡 Interpretation
            </h2>

            <div style="background: #e6fffa; border-left: 5px solid #38b2ac; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    No obvious fractures detected in this X-ray image.<br>
                    However, subtle fractures may not be detected by AI.
                </p>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                🏥 Recommendation
            </h2>

            <div style="background: #fffff0; border-left: 5px solid #ecc94b; padding: 15px; margin: 15px 0; border-radius: 5px;">
                <p style="margin: 0; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    <strong>If you experience persistent pain or symptoms:</strong>
                </p>
                <ul style="margin: 10px 0; padding-left: 20px; font-size: 15px; color: #2d3748; line-height: 1.6;">
                    <li>Consult a medical professional</li>
                    <li>Clinical examination may reveal findings not visible on X-ray</li>
                    <li>Follow-up imaging may be recommended</li>
                </ul>
            </div>

            <h2 style="color: #2d3748; font-size: 22px; margin-top: 25px; border-bottom: 3px solid #48bb78; padding-bottom: 10px;">
                ⚠️ Important Disclaimer
            </h2>

            <div style="background: #fffaf0; border: 2px solid #ed8936; padding: 15px; border-radius: 8px; margin-top: 15px;">
                <p style="margin: 0; font-size: 14px; color: #2d3748; line-height: 1.6;">
                    This is an AI screening tool. Normal AI results do not rule out fractures.<br>
                    Always consult a qualified medical professional for proper diagnosis and treatment.
                </p>
            </div>
        </div>
        """
    
    return annotated_img, result_html, has_fracture, num_detections

print("✅ Prediction function defined!")
print("   - Uses YOLOv8 for fast, accurate detection")
print("   - Returns annotated image and HTML report")
print("="*70)

DEFINING PREDICTION FUNCTION
✅ Prediction function defined!
   - Uses YOLOv8 for fast, accurate detection
   - Returns annotated image and HTML report


In [7]:
# ============================================================================
# BLOCK 4: GRADIO WEB INTERFACE (Professional UI)
# ============================================================================

print("="*70)
print("LAUNCHING GRADIO WEB INTERFACE")
print("="*70)

import gradio as gr

def gradio_predict(image, threshold):
    """Wrapper for Gradio interface"""
    if image is None:
        return None, "Please upload an image."
    
    # Convert Gradio image to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Run prediction
    annotated_img, result_text, has_fracture, num_fractures = predict_fracture(image, threshold)
    
    return annotated_img, result_text

# Create Gradio interface with your exact design
with gr.Blocks(theme=gr.themes.Soft(), title="AETHEA Shoulder & Arm Fracture Detection") as demo:
    gr.Markdown("""
    # 🏥 AETHEA Shoulder & Arm Fracture Detection System
    
    Upload an X-ray image of a **shoulder or arm** to detect potential fractures using AI.
    
    **Model Performance:** mAP50: 87.6% | Precision: 89.9% | Recall: 85.2% | YOLOv8m
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload Shoulder/Arm X-ray", height=400)
            
            threshold = gr.Slider(
                minimum=0.1,
                maximum=0.9,
                value=0.3,
                step=0.05,
                label="Detection Threshold",
                info="Lower = more sensitive (more detections), Higher = more specific (fewer false positives)"
            )
            
            submit_btn = gr.Button("🔍 Detect Fracture", variant="primary", size="lg")
        
        with gr.Column(scale=1):
            image_output = gr.Image(label="Analysis Result", height=400)
    
    with gr.Row():
        result_output = gr.HTML(label="Diagnosis Report")
    
    submit_btn.click(
        fn=gradio_predict,
        inputs=[image_input, threshold],
        outputs=[image_output, result_output]
    )
    
    gr.Markdown("""
    ---
    ### 📋 Instructions
    1. Upload a shoulder or arm X-ray image (JPG, PNG)
    2. Adjust threshold if needed (0.3 is recommended)
    3. Click "Detect Fracture" to analyze
    
    ### 🎯 Threshold Guide
    | Threshold | Sensitivity | Use Case |
    |-----------|-------------|----------|
    | 0.2 | High | Catch subtle fractures (more false positives) |
    | 0.3 | Balanced | Default recommendation |
    | 0.5 | Low | Only confident detections |
    
    ### 📊 Model Performance
    | Metric | Value |
    |--------|-------|
    | mAP50 | 87.6% |
    | Precision | 89.9% |
    | Recall | 85.2% |
    | Inference Speed | ~0.03s/image |
    
    ### ⚕️ Medical Disclaimer
    This is an AI screening tool. All results must be verified by qualified healthcare professionals.
    """)

# Launch
demo.launch(share=True, debug=False)

print("\n✅ Gradio interface launched!")
print("📤 Share the public link above with others")
print("="*70)

LAUNCHING GRADIO WEB INTERFACE
* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://4668930da67212ad26.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Gradio interface launched!
📤 Share the public link above with others


In [15]:
# ============================================================================
# IMPORTS FOR ALL ANALYSIS BLOCKS
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
from tqdm import tqdm
from ultralytics import YOLO
import pandas as pd

print("✅ All imports loaded")

✅ All imports loaded


In [17]:
# ============================================================================
# CONFIDENCE THRESHOLD ANALYSIS - ARM & SHOULDER MODEL
# Finds optimal threshold for best precision/recall balance
# ============================================================================

print("="*70)
print("CONFIDENCE THRESHOLD ANALYSIS - ARM & SHOULDER")
print("="*70)

# Paths
MODEL_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt"
TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\images"
TEST_LABELS_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\labels"
OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load model
model = YOLO(MODEL_PATH)

def compute_precision_recall_vs_threshold(model, images_dir, labels_dir, thresholds):
    """Compute precision and recall at different confidence thresholds"""
    
    # Convert to Path objects
    images_path = Path(images_dir)
    labels_path = Path(labels_dir)
    
    # Get all image files
    image_files = list(images_path.glob("*.jpg")) + list(images_path.glob("*.png"))
    
    # Store all predictions and ground truths
    all_predictions = []  # Store max confidence per image
    all_ground_truths = []  # Store whether image has fracture
    
    print(f"📊 Processing {len(image_files)} images...")
    
    for img_path in tqdm(image_files, desc="Evaluating"):
        label_path = labels_path / f"{img_path.stem}.txt"
        
        # Run inference
        results = model(str(img_path), verbose=False)
        boxes = results[0].boxes
        
        # Get max confidence if any detection
        if boxes is not None and len(boxes) > 0:
            max_conf = float(boxes.conf.cpu().numpy().max())
            all_predictions.append(max_conf)
        else:
            all_predictions.append(0.0)
        
        # Get ground truth (has fracture?)
        has_fracture = False
        if label_path.exists():
            with open(label_path, 'r') as f:
                content = f.read().strip()
                if content and len(content) > 0:
                    has_fracture = True
        all_ground_truths.append(has_fracture)
    
    # Compute metrics at each threshold
    precisions = []
    recalls = []
    f1_scores = []
    
    for thresh in thresholds:
        tp = fp = fn = tn = 0
        
        for pred_conf, has_gt in zip(all_predictions, all_ground_truths):
            pred_positive = pred_conf >= thresh
            
            if pred_positive and has_gt:
                tp += 1
            elif pred_positive and not has_gt:
                fp += 1
            elif not pred_positive and has_gt:
                fn += 1
            else:
                tn += 1
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision * 100)
        recalls.append(recall * 100)
        f1_scores.append(f1 * 100)
    
    return precisions, recalls, f1_scores, all_predictions, all_ground_truths

# Test thresholds
thresholds = np.arange(0.05, 0.95, 0.05)

print(f"📊 Testing across {len(thresholds)} thresholds...")

precisions, recalls, f1_scores, all_confs, all_gts = compute_precision_recall_vs_threshold(
    model, TEST_IMAGES_PATH, TEST_LABELS_PATH, thresholds
)

# Find optimal threshold (max F1 score)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_precision = precisions[best_idx]
best_recall = recalls[best_idx]

print(f"\n📈 RESULTS:")
print(f"   Optimal threshold: {best_threshold:.2f}")
print(f"   F1 Score: {best_f1:.1f}%")
print(f"   Precision at optimal: {best_precision:.1f}%")
print(f"   Recall at optimal: {best_recall:.1f}%")

# Create publication-quality figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Plot 1: Precision/Recall vs Threshold
ax1 = axes[0]
ax1.plot(thresholds, precisions, 'b-', linewidth=2.5, label='Precision', marker='s', markersize=6)
ax1.plot(thresholds, recalls, 'g-', linewidth=2.5, label='Recall', marker='o', markersize=6)
ax1.plot(thresholds, f1_scores, 'r--', linewidth=2.5, label='F1 Score', marker='^', markersize=6)
ax1.axvline(x=best_threshold, color='purple', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.scatter(best_threshold, best_f1, color='purple', s=100, zorder=5, edgecolors='black')
ax1.set_xlabel('Confidence Threshold', fontsize=12, fontweight='bold')
ax1.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax1.set_title('Arm & Shoulder: Precision/Recall vs Threshold', fontsize=14, fontweight='bold')
ax1.set_xlim(0.05, 0.95)
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(loc='best', fontsize=10)
ax1.text(0.02, 0.95, f'Optimal: {best_threshold:.2f}\nF1: {best_f1:.1f}%', 
         transform=ax1.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 2: Precision-Recall Curve
ax2 = axes[1]
ax2.plot(recalls, precisions, 'b-', linewidth=2.5, marker='o', markersize=6)
ax2.scatter(best_recall, best_precision, color='red', s=100, zorder=5, edgecolors='black')
ax2.set_xlabel('Recall (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Precision (%)', fontsize=12, fontweight='bold')
ax2.set_title('Arm & Shoulder: Precision-Recall Curve', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 100)
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.text(0.02, 0.05, f'Optimal Point\nP: {best_precision:.1f}% | R: {best_recall:.1f}%', 
         transform=ax2.transAxes, fontsize=10, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'arm_shoulder_threshold_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {os.path.join(OUTPUT_DIR, 'arm_shoulder_threshold_analysis.png')}")

CONFIDENCE THRESHOLD ANALYSIS - ARM & SHOULDER
📊 Testing across 18 thresholds...
📊 Processing 549 images...


Evaluating: 100%|████████████████████████████████████████████████████████████████████| 549/549 [00:35<00:00, 15.65it/s]



📈 RESULTS:
   Optimal threshold: 0.05
   F1 Score: 99.4%
   Precision at optimal: 100.0%
   Recall at optimal: 98.7%


<Figure size 1400x500 with 2 Axes>

✅ Saved: C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\arm_shoulder_threshold_analysis.png


In [19]:
# ============================================================================
# MONTAGE GENERATOR - Creates grid of test images with bounding boxes
# ============================================================================

def create_montage(model, images_dir, output_path, grid_size=(3, 4), threshold=0.3, 
                   max_images=12, title="Fracture Detection Results"):
    """
    Create a grid montage of images with bounding boxes
    """
    # Get image files
    images_path = Path(images_dir)
    image_files = list(images_path.glob("*.jpg")) + list(images_path.glob("*.png"))
    image_files = image_files[:max_images]
    
    rows, cols = grid_size
    n_images = len(image_files)
    
    # Calculate figure size
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.98)
    fig.patch.set_facecolor('white')
    
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for idx, img_path in enumerate(image_files):
        # Run prediction
        results = model(str(img_path), conf=threshold, verbose=False)
        annotated_img = results[0].plot()
        
        # Get detection info
        boxes = results[0].boxes
        num_detections = len(boxes) if boxes else 0
        
        # Display image
        axes[idx].imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
        
        # Add title with detection info
        if num_detections > 0:
            confidences = boxes.conf.cpu().numpy()
            max_conf = max(confidences) * 100
            axes[idx].set_title(f'⚠️ Fracture | Conf: {max_conf:.1f}%', 
                               color='red', fontweight='bold', fontsize=10)
        else:
            axes[idx].set_title('✅ No Fracture', color='green', fontweight='bold', fontsize=10)
        
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Montage saved: {output_path}")

# For Arm & Shoulder
print("\n" + "="*70)
print("GENERATING ARM & SHOULDER MONTAGE")
print("="*70)

model = YOLO(r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt")
OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\images"

create_montage(
    model, TEST_IMAGES_PATH,
    os.path.join(OUTPUT_DIR, 'arm_shoulder_montage.png'),
    grid_size=(3, 4), max_images=12, threshold=0.3,
    title="Arm & Shoulder: Fracture Detection Results (Threshold: 0.3)"
)

# For Hips & Pelvis
print("\n" + "="*70)
print("GENERATING HIPS & PELVIS MONTAGE")
print("="*70)

model = YOLO(r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt")
OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\images"

create_montage(
    model, TEST_IMAGES_PATH,
    os.path.join(OUTPUT_DIR, 'hips_pelvis_montage.png'),
    grid_size=(3, 4), max_images=12, threshold=0.3,
    title="Hips & Pelvis: Fracture Detection Results (Threshold: 0.3)"
)

print("\n✅ All montages generated!")


GENERATING ARM & SHOULDER MONTAGE
✅ Montage saved: C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\arm_shoulder_montage.png

GENERATING HIPS & PELVIS MONTAGE


C:\Users\andre\AppData\Local\Temp\ipykernel_26980\1651460861.py:53: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\andre\AppData\Local\Temp\ipykernel_26980\1651460861.py:54: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(output_path, dpi=300, bbox_inches='tight')


✅ Montage saved: C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\hips_pelvis_montage.png

✅ All montages generated!


In [21]:
# ============================================================================
# CONFIDENCE DISTRIBUTION HISTOGRAM
# ============================================================================

def plot_confidence_distribution(model, images_dir, output_path, title, max_images=200):
    """Plot histogram of confidence scores"""
    
    images_path = Path(images_dir)
    image_files = list(images_path.glob("*.jpg")) + list(images_path.glob("*.png"))
    image_files = image_files[:max_images]
    
    all_confidences = []
    
    for img_path in tqdm(image_files, desc="Processing"):
        results = model(str(img_path), verbose=False)
        boxes = results[0].boxes
        
        if boxes is not None and len(boxes) > 0:
            confidences = boxes.conf.cpu().numpy()
            all_confidences.extend(confidences)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('white')
    
    if len(all_confidences) > 0:
        ax.hist(all_confidences, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        ax.axvline(x=0.3, color='red', linestyle='--', linewidth=2, label='Default Threshold (0.3)')
        ax.axvline(x=np.mean(all_confidences), color='green', linestyle='--', linewidth=2, 
                   label=f'Mean Confidence: {np.mean(all_confidences)*100:.1f}%')
    else:
        ax.text(0.5, 0.5, 'No detections found', ha='center', va='center', fontsize=14)
    
    ax.set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Saved: {output_path}")
    if len(all_confidences) > 0:
        print(f"   Number of detections: {len(all_confidences)}")
        print(f"   Mean confidence: {np.mean(all_confidences)*100:.1f}%")
        print(f"   Median confidence: {np.median(all_confidences)*100:.1f}%")
        print(f"   Std confidence: {np.std(all_confidences)*100:.1f}%")

# For Arm & Shoulder
print("\n" + "="*70)
print("ARM & SHOULDER - CONFIDENCE DISTRIBUTION")
print("="*70)

OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results"
model = YOLO(r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt")
TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\images"

plot_confidence_distribution(
    model, TEST_IMAGES_PATH,
    os.path.join(OUTPUT_DIR, 'arm_shoulder_confidence_distribution.png'),
    "Arm & Shoulder: Detection Confidence Distribution"
)

# For Hips & Pelvis
print("\n" + "="*70)
print("HIPS & PELVIS - CONFIDENCE DISTRIBUTION")
print("="*70)

OUTPUT_DIR = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results"
model = YOLO(r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt")
TEST_IMAGES_PATH = r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\images"

plot_confidence_distribution(
    model, TEST_IMAGES_PATH,
    os.path.join(OUTPUT_DIR, 'hips_pelvis_confidence_distribution.png'),
    "Hips & Pelvis: Detection Confidence Distribution"
)

print("\n✅ All analysis complete!")
print("   Check the Analysis_Results folders for output images.")


ARM & SHOULDER - CONFIDENCE DISTRIBUTION


Processing: 100%|████████████████████████████████████████████████████████████████████| 200/200 [00:11<00:00, 16.82it/s]


<Figure size 1000x600 with 1 Axes>

✅ Saved: C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\arm_shoulder_confidence_distribution.png
   Number of detections: 299
   Mean confidence: 64.8%
   Median confidence: 64.2%
   Std confidence: 22.2%

HIPS & PELVIS - CONFIDENCE DISTRIBUTION


Processing: 100%|████████████████████████████████████████████████████████████████████| 194/194 [00:10<00:00, 18.22it/s]


<Figure size 1000x600 with 1 Axes>

✅ Saved: C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\hips_pelvis_confidence_distribution.png
   Number of detections: 337
   Mean confidence: 76.5%
   Median confidence: 85.5%
   Std confidence: 18.4%

✅ All analysis complete!
   Check the Analysis_Results folders for output images.


In [23]:
# ============================================================================
# AETHEA - PROFESSIONAL ANALYSIS FIGURES FOR SCIENTIFIC PAPER
# Generates 5 publication-quality figures for both models
#
# HOW TO USE:
#   1. Open Anaconda Prompt
#   2. pip install ultralytics opencv-python matplotlib numpy tqdm
#   3. python generate_analysis_figures.py
#   4. Find all figures in the Analysis_Results folders
# ============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import cv2
from tqdm import tqdm
from ultralytics import YOLO

# ============================================================================
# *** CONFIGURATION — UPDATE THESE PATHS ***
# ============================================================================

MODELS = {
    'arm_shoulder': {
        'model_path':  r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt",
        'test_images': r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\images",
        'test_labels': r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\labels",
        'output_dir':  r"C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results",
        'label':       'Arm & Shoulder',
        'color':       '#1B4F8A',
        'color2':      '#1ABC9C',
    },
    'hips_pelvis': {
        'model_path':  r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt",
        'test_images': r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\images",
        'test_labels': r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\test\labels",
        'output_dir':  r"C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results",
        'label':       'Hips & Pelvis',
        'color':       '#8E44AD',
        'color2':      '#E67E22',
    },
}

MAX_IMAGES_MONTAGE   = 12    # Images to show in montage grid
MAX_IMAGES_HISTOGRAM = 300   # Images to scan for confidence histogram
THRESHOLD_RANGE      = np.arange(0.05, 0.96, 0.05)

# ============================================================================
# HELPERS
# ============================================================================

def load_model(path):
    print(f"   Loading model: {path}")
    return YOLO(path)


def get_image_files(images_dir, limit=None):
    p = Path(images_dir)
    files = list(p.glob("*.jpg")) + list(p.glob("*.jpeg")) + list(p.glob("*.png"))
    if limit:
        files = files[:limit]
    return files


def compute_pr_vs_threshold(model, images_dir, labels_dir, thresholds):
    """Return precision, recall, f1 arrays across thresholds."""
    image_files = get_image_files(images_dir)
    labels_path = Path(labels_dir)

    all_confs, all_gts = [], []

    for img_path in tqdm(image_files, desc="   Scanning images", leave=False):
        results = model(str(img_path), verbose=False)
        boxes   = results[0].boxes
        conf    = float(boxes.conf.cpu().numpy().max()) if boxes is not None and len(boxes) > 0 else 0.0
        all_confs.append(conf)

        label_path = labels_path / f"{img_path.stem}.txt"
        has_fracture = label_path.exists() and label_path.read_text().strip() != ""
        all_gts.append(has_fracture)

    precisions, recalls, f1s = [], [], []
    for thresh in thresholds:
        tp = fp = fn = tn = 0
        for conf, gt in zip(all_confs, all_gts):
            pos = conf >= thresh
            if pos and gt:     tp += 1
            elif pos:          fp += 1
            elif gt:           fn += 1
            else:              tn += 1
        p  = tp / (tp + fp) if tp + fp > 0 else 0
        r  = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2*p*r / (p+r)  if p + r  > 0 else 0
        precisions.append(p * 100)
        recalls.append(r * 100)
        f1s.append(f1 * 100)

    return np.array(precisions), np.array(recalls), np.array(f1s), all_confs, all_gts


# ============================================================================
# FIGURE 1 — CONFIDENCE THRESHOLD ANALYSIS
# Best threshold clearly marked. Two subplots.
# ============================================================================

def fig1_threshold_analysis(cfg, model):
    print(f"\n[Figure 1] Threshold analysis — {cfg['label']}")

    prec, rec, f1, confs, gts = compute_pr_vs_threshold(
        model, cfg['test_images'], cfg['test_labels'], THRESHOLD_RANGE)

    best_idx  = int(np.argmax(f1))
    best_thr  = THRESHOLD_RANGE[best_idx]
    best_f1   = f1[best_idx]
    best_prec = prec[best_idx]
    best_rec  = rec[best_idx]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.patch.set_facecolor('white')
    fig.suptitle(f"{cfg['label']} — Confidence Threshold Analysis",
                 fontsize=15, fontweight='bold', y=1.01)

    # — Left: P / R / F1 vs threshold —
    ax = axes[0]
    ax.fill_between(THRESHOLD_RANGE, prec, alpha=0.08, color='#2980B9')
    ax.fill_between(THRESHOLD_RANGE, rec,  alpha=0.08, color='#27AE60')
    ax.plot(THRESHOLD_RANGE, prec, 'o-', color='#2980B9', lw=2.5, ms=6,  label='Precision')
    ax.plot(THRESHOLD_RANGE, rec,  's-', color='#27AE60', lw=2.5, ms=6,  label='Recall')
    ax.plot(THRESHOLD_RANGE, f1,   '^--',color='#E74C3C', lw=2.5, ms=6,  label='F1 Score')

    # Optimal marker
    ax.axvline(best_thr, color='#8E44AD', ls='--', lw=1.8, alpha=0.8)
    ax.scatter(best_thr, best_f1, color='#8E44AD', s=130, zorder=6,
               edgecolors='white', linewidths=1.5, label=f'Optimal ({best_thr:.2f})')
    ax.annotate(f'Best threshold\n{best_thr:.2f}  →  F1 {best_f1:.1f}%',
                xy=(best_thr, best_f1),
                xytext=(best_thr + 0.07, best_f1 - 18),
                fontsize=9, color='#8E44AD', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=1.3))

    ax.set_xlabel('Confidence Threshold', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)',             fontsize=12, fontweight='bold')
    ax.set_title('Precision / Recall / F1 vs Threshold', fontsize=12)
    ax.set_xlim(0.03, 0.97); ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3, ls='--')
    ax.legend(fontsize=10, loc='lower left')

    # — Right: Precision-Recall curve —
    ax2 = axes[1]
    # Color the curve by threshold value
    for i in range(len(THRESHOLD_RANGE) - 1):
        ax2.plot(rec[i:i+2], prec[i:i+2],
                 color=plt.cm.RdYlGn(i / len(THRESHOLD_RANGE)), lw=2.5)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap='RdYlGn',
                                norm=plt.Normalize(THRESHOLD_RANGE[0], THRESHOLD_RANGE[-1]))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax2, shrink=0.7, pad=0.02)
    cbar.set_label('Confidence Threshold', fontsize=9)

    ax2.scatter(best_rec, best_prec, color='#8E44AD', s=160, zorder=6,
                edgecolors='white', linewidths=2, label=f'Optimal point')
    ax2.annotate(f'  P={best_prec:.1f}%\n  R={best_rec:.1f}%',
                 xy=(best_rec, best_prec), fontsize=9, color='#8E44AD', fontweight='bold')

    ax2.set_xlabel('Recall (%)',    fontsize=12, fontweight='bold')
    ax2.set_ylabel('Precision (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Precision-Recall Curve\n(colour = threshold)', fontsize=12)
    ax2.set_xlim(0, 105); ax2.set_ylim(0, 105)
    ax2.grid(True, alpha=0.3, ls='--')
    ax2.legend(fontsize=10)

    plt.tight_layout()
    out = os.path.join(cfg['output_dir'], 'fig1_threshold_analysis.png')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Saved → {out}")
    print(f"   Best threshold: {best_thr:.2f}  |  F1: {best_f1:.1f}%  |  P: {best_prec:.1f}%  |  R: {best_rec:.1f}%")
    return best_thr


# ============================================================================
# FIGURE 2 — DETECTION MONTAGE (grid of images with bounding boxes)
# ============================================================================

def fig2_detection_montage(cfg, model, threshold=0.3):
    print(f"\n[Figure 2] Detection montage — {cfg['label']}")

    image_files = get_image_files(cfg['test_images'])
    # Pick a mix: some with detections, some without
    with_det, without_det = [], []
    for img_path in image_files:
        if len(with_det) >= 8 and len(without_det) >= 4:
            break
        res = model(str(img_path), conf=threshold, verbose=False)
        if res[0].boxes is not None and len(res[0].boxes) > 0 and len(with_det) < 8:
            with_det.append(img_path)
        elif len(without_det) < 4:
            without_det.append(img_path)

    selected = with_det + without_det          # up to 12 images
    n        = len(selected)
    cols     = 4
    rows     = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4.2))
    fig.patch.set_facecolor('#F0F4F8')
    fig.suptitle(f"{cfg['label']} — Fracture Detection Results\n"
                 f"(Confidence threshold: {threshold:.2f}  |  "
                 f"Red box = fracture detected  |  Green title = no fracture)",
                 fontsize=13, fontweight='bold', y=1.01)

    axes_flat = axes.flatten() if rows * cols > 1 else [axes]

    for idx, img_path in enumerate(selected):
        res           = model(str(img_path), conf=threshold, verbose=False)
        annotated     = res[0].plot(line_width=2)
        boxes         = res[0].boxes
        num_det       = len(boxes) if boxes is not None else 0

        ax = axes_flat[idx]
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

        if num_det > 0:
            confs = boxes.conf.cpu().numpy()
            ax.set_title(f'⚠️  Fracture  |  conf {max(confs)*100:.1f}%',
                         color='#C0392B', fontweight='bold', fontsize=10, pad=4)
            for spine in ax.spines.values():
                spine.set_edgecolor('#C0392B'); spine.set_linewidth(3)
        else:
            ax.set_title('✅  No Fracture',
                         color='#1E8449', fontweight='bold', fontsize=10, pad=4)
            for spine in ax.spines.values():
                spine.set_edgecolor('#1E8449'); spine.set_linewidth(3)

        ax.set_xticks([]); ax.set_yticks([])

    for idx in range(len(selected), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    plt.tight_layout()
    out = os.path.join(cfg['output_dir'], 'fig2_detection_montage.png')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='#F0F4F8')
    plt.close()
    print(f"   ✅ Saved → {out}")


# ============================================================================
# FIGURE 3 — CONFIDENCE SCORE DISTRIBUTION HISTOGRAM
# ============================================================================

def fig3_confidence_histogram(cfg, model):
    print(f"\n[Figure 3] Confidence histogram — {cfg['label']}")

    image_files = get_image_files(cfg['test_images'], limit=MAX_IMAGES_HISTOGRAM)
    all_confs   = []

    for img_path in tqdm(image_files, desc="   Scanning", leave=False):
        res   = model(str(img_path), verbose=False)
        boxes = res[0].boxes
        if boxes is not None and len(boxes) > 0:
            all_confs.extend(boxes.conf.cpu().numpy().tolist())

    fig, ax = plt.subplots(figsize=(10, 5.5))
    fig.patch.set_facecolor('white')

    if len(all_confs) > 0:
        n, bins, patches = ax.hist(all_confs, bins=30, edgecolor='white', linewidth=0.8)

        # Color bars by height
        norm = plt.Normalize(n.min(), n.max())
        cmap = plt.cm.get_cmap('Blues')
        for patch, val in zip(patches, n):
            patch.set_facecolor(cmap(norm(val)))

        mean_c  = np.mean(all_confs)
        median_c= np.median(all_confs)

        ax.axvline(0.30,     color='#E74C3C', ls='--', lw=2,   label='Default threshold (0.30)')
        ax.axvline(mean_c,   color='#27AE60', ls='-.',  lw=2,   label=f'Mean: {mean_c*100:.1f}%')
        ax.axvline(median_c, color='#F39C12', ls=':',   lw=2,   label=f'Median: {median_c*100:.1f}%')

        # Stats box
        stats = (f"Detections: {len(all_confs)}\n"
                 f"Mean:   {mean_c*100:.1f}%\n"
                 f"Median: {median_c*100:.1f}%\n"
                 f"Std:    {np.std(all_confs)*100:.1f}%")
        ax.text(0.98, 0.97, stats, transform=ax.transAxes, fontsize=10,
                va='top', ha='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#EBF5FB', alpha=0.9))
    else:
        ax.text(0.5, 0.5, 'No detections found', ha='center', va='center',
                fontsize=14, transform=ax.transAxes)

    ax.set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Detections', fontsize=12, fontweight='bold')
    ax.set_title(f"{cfg['label']} — Detection Confidence Distribution",
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, ls='--', axis='y')
    ax.set_xlim(0, 1)

    plt.tight_layout()
    out = os.path.join(cfg['output_dir'], 'fig3_confidence_histogram.png')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Saved → {out}")


# ============================================================================
# FIGURE 4 — TP / FP / FN BREAKDOWN BAR CHART AT BEST THRESHOLD
# ============================================================================

def fig4_tp_fp_fn_breakdown(cfg, model, threshold):
    print(f"\n[Figure 4] TP/FP/FN breakdown — {cfg['label']}")

    image_files = get_image_files(cfg['test_images'])
    labels_path = Path(cfg['test_labels'])

    tp = fp = fn = tn = 0
    for img_path in tqdm(image_files, desc="   Evaluating", leave=False):
        res   = model(str(img_path), conf=threshold, verbose=False)
        boxes = res[0].boxes
        detected = boxes is not None and len(boxes) > 0

        label_path   = labels_path / f"{img_path.stem}.txt"
        has_fracture = label_path.exists() and label_path.read_text().strip() != ""

        if detected and has_fracture:     tp += 1
        elif detected and not has_fracture: fp += 1
        elif not detected and has_fracture: fn += 1
        else:                               tn += 1

    total  = tp + fp + fn + tn
    cats   = ['True\nPositive', 'True\nNegative', 'False\nPositive', 'False\nNegative']
    vals   = [tp, tn, fp, fn]
    colors = ['#1ABC9C', '#2980B9', '#E74C3C', '#E67E22']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.patch.set_facecolor('white')
    fig.suptitle(f"{cfg['label']} — Detection Breakdown  (threshold = {threshold:.2f})",
                 fontsize=13, fontweight='bold')

    # Bar chart
    ax = axes[0]
    bars = ax.bar(cats, vals, color=colors, width=0.55, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
    ax.set_title('Classification Results', fontsize=12)
    ax.set_ylim(0, max(vals) * 1.25)
    ax.grid(True, alpha=0.3, ls='--', axis='y')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    # Pie chart
    ax2 = axes[1]
    wedges, texts, autotexts = ax2.pie(
        vals, labels=cats, colors=colors, autopct='%1.1f%%',
        startangle=140, pctdistance=0.75,
        wedgeprops=dict(edgecolor='white', linewidth=2))
    for at in autotexts:
        at.set_fontsize(10); at.set_fontweight('bold')
    ax2.set_title('Proportion Breakdown', fontsize=12)

    # Derived metrics box
    precision = tp / (tp + fp) * 100 if tp + fp > 0 else 0
    recall    = tp / (tp + fn) * 100 if tp + fn > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    accuracy  = (tp + tn) / total * 100

    metrics_text = (f"Precision : {precision:.1f}%\n"
                    f"Recall    : {recall:.1f}%\n"
                    f"F1 Score  : {f1:.1f}%\n"
                    f"Accuracy  : {accuracy:.1f}%\n"
                    f"Total images: {total}")
    ax2.text(0, -1.35, metrics_text, ha='center', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.6', facecolor='#EBF5FB', alpha=0.9))

    plt.tight_layout()
    out = os.path.join(cfg['output_dir'], 'fig4_tp_fp_fn_breakdown.png')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"   ✅ Saved → {out}")
    print(f"   Precision:{precision:.1f}%  Recall:{recall:.1f}%  F1:{f1:.1f}%  Acc:{accuracy:.1f}%")


# ============================================================================
# FIGURE 5 — SIDE-BY-SIDE COMPARISON: GROUND TRUTH vs PREDICTION
# ============================================================================

def fig5_gt_vs_pred(cfg, model, threshold=0.3, n_pairs=6):
    print(f"\n[Figure 5] GT vs Prediction — {cfg['label']}")

    image_files = get_image_files(cfg['test_images'])
    labels_path = Path(cfg['test_labels'])

    # Pick images that HAVE ground truth labels
    pairs = []
    for img_path in image_files:
        if len(pairs) >= n_pairs:
            break
        label_path = labels_path / f"{img_path.stem}.txt"
        if label_path.exists() and label_path.read_text().strip():
            pairs.append(img_path)

    cols = 2 * n_pairs if n_pairs <= 3 else n_pairs
    rows = 2

    fig, axes = plt.subplots(rows, n_pairs, figsize=(n_pairs * 4.5, 9))
    fig.patch.set_facecolor('#1C2833')
    fig.suptitle(f"{cfg['label']} — Ground Truth (top) vs Model Prediction (bottom)",
                 fontsize=13, fontweight='bold', color='white', y=1.01)

    for col, img_path in enumerate(pairs):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w    = img_bgr.shape[:2]

        # — Top row: Ground Truth —
        ax_gt = axes[0][col]
        gt_img = img_rgb.copy()
        label_path = labels_path / f"{img_path.stem}.txt"
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    _, cx, cy, bw, bh = map(float, parts[:5])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 200, 100), 3)
                    cv2.putText(gt_img, 'GT', (x1, max(y1-8, 12)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 100), 2)

        ax_gt.imshow(gt_img)
        ax_gt.set_title('Ground Truth', color='#1ABC9C', fontweight='bold', fontsize=11)
        ax_gt.set_xticks([]); ax_gt.set_yticks([])
        for sp in ax_gt.spines.values():
            sp.set_edgecolor('#1ABC9C'); sp.set_linewidth(2.5)

        # — Bottom row: Prediction —
        ax_pred = axes[1][col]
        res      = model(str(img_path), conf=threshold, verbose=False)
        pred_img = res[0].plot(line_width=2)
        boxes    = res[0].boxes
        num_det  = len(boxes) if boxes is not None else 0

        ax_pred.imshow(cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB))
        if num_det > 0:
            confs = boxes.conf.cpu().numpy()
            ax_pred.set_title(f'Predicted  ✓  {max(confs)*100:.1f}%',
                              color='#E74C3C', fontweight='bold', fontsize=11)
            for sp in ax_pred.spines.values():
                sp.set_edgecolor('#E74C3C'); sp.set_linewidth(2.5)
        else:
            ax_pred.set_title('Predicted  ✗  Missed',
                              color='#F39C12', fontweight='bold', fontsize=11)
            for sp in ax_pred.spines.values():
                sp.set_edgecolor('#F39C12'); sp.set_linewidth(2.5)
        ax_pred.set_xticks([]); ax_pred.set_yticks([])

    plt.tight_layout()
    out = os.path.join(cfg['output_dir'], 'fig5_gt_vs_prediction.png')
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='#1C2833')
    plt.close()
    print(f"   ✅ Saved → {out}")


# ============================================================================
# MAIN — RUN ALL FIGURES FOR BOTH MODELS
# ============================================================================

def main():
    print("=" * 70)
    print("  AETHEA — GENERATING ALL ANALYSIS FIGURES")
    print("  For: Arm & Shoulder  +  Hips & Pelvis")
    print("=" * 70)

    for model_key, cfg in MODELS.items():
        print(f"\n{'='*70}")
        print(f"  MODEL: {cfg['label'].upper()}")
        print(f"{'='*70}")

        os.makedirs(cfg['output_dir'], exist_ok=True)

        model = load_model(cfg['model_path'])

        # Figure 1 — returns best threshold for use in other figures
        best_thr = fig1_threshold_analysis(cfg, model)

        # Figure 2 — detection montage using best threshold
        fig2_detection_montage(cfg, model, threshold=best_thr)

        # Figure 3 — confidence histogram
        fig3_confidence_histogram(cfg, model)

        # Figure 4 — TP/FP/FN breakdown
        fig4_tp_fp_fn_breakdown(cfg, model, threshold=best_thr)

        # Figure 5 — GT vs Prediction comparison
        fig5_gt_vs_pred(cfg, model, threshold=best_thr, n_pairs=6)

        print(f"\n  ✅ {cfg['label']} — all 5 figures saved to:")
        print(f"     {cfg['output_dir']}")

    print("\n" + "=" * 70)
    print("  ✅ ALL FIGURES GENERATED SUCCESSFULLY!")
    print("=" * 70)
    print("\n  Files per model:")
    print("  fig1_threshold_analysis.png   — P/R/F1 curves + optimal threshold")
    print("  fig2_detection_montage.png    — Grid of test images with bboxes")
    print("  fig3_confidence_histogram.png — Confidence score distribution")
    print("  fig4_tp_fp_fn_breakdown.png   — Bar + pie chart of TP/FP/FN/TN")
    print("  fig5_gt_vs_prediction.png     — Ground truth vs model predictions")
    print("=" * 70)


if __name__ == '__main__':
    main()

  AETHEA — GENERATING ALL ANALYSIS FIGURES
  For: Arm & Shoulder  +  Hips & Pelvis

  MODEL: ARM & SHOULDER
   Loading model: C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt

[Figure 1] Threshold analysis — Arm & Shoulder


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\fig1_threshold_analysis.png
   Best threshold: 0.05  |  F1: 99.4%  |  P: 100.0%  |  R: 98.7%

[Figure 2] Detection montage — Arm & Shoulder
   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\fig2_detection_montage.png

[Figure 3] Confidence histogram — Arm & Shoulder


C:\Users\andre\AppData\Local\Temp\ipykernel_26980\34783448.py:277: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('Blues')


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\fig3_confidence_histogram.png

[Figure 4] TP/FP/FN breakdown — Arm & Shoulder


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\fig4_tp_fp_fn_breakdown.png
   Precision:100.0%  Recall:99.8%  F1:99.9%  Acc:99.8%

[Figure 5] GT vs Prediction — Arm & Shoulder
   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results\fig5_gt_vs_prediction.png

  ✅ Arm & Shoulder — all 5 figures saved to:
     C:\Users\andre\Documents\Graduation Project\Shoulders\Analysis_Results

  MODEL: HIPS & PELVIS
   Loading model: C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\NEW YOLO 38 Epochs\best.pt

[Figure 1] Threshold analysis — Hips & Pelvis


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\fig1_threshold_analysis.png
   Best threshold: 0.05  |  F1: 95.7%  |  P: 100.0%  |  R: 91.8%

[Figure 2] Detection montage — Hips & Pelvis


C:\Users\andre\AppData\Local\Temp\ipykernel_26980\34783448.py:246: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\andre\AppData\Local\Temp\ipykernel_26980\34783448.py:248: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='#F0F4F8')


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\fig2_detection_montage.png

[Figure 3] Confidence histogram — Hips & Pelvis


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\fig3_confidence_histogram.png

[Figure 4] TP/FP/FN breakdown — Hips & Pelvis


   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\fig4_tp_fp_fn_breakdown.png
   Precision:100.0%  Recall:99.0%  F1:99.5%  Acc:99.0%

[Figure 5] GT vs Prediction — Hips & Pelvis
   ✅ Saved → C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results\fig5_gt_vs_prediction.png

  ✅ Hips & Pelvis — all 5 figures saved to:
     C:\Users\andre\Documents\Graduation Project\Hips and Pelvis\Analysis_Results

  ✅ ALL FIGURES GENERATED SUCCESSFULLY!

  Files per model:
  fig1_threshold_analysis.png   — P/R/F1 curves + optimal threshold
  fig2_detection_montage.png    — Grid of test images with bboxes
  fig3_confidence_histogram.png — Confidence score distribution
  fig4_tp_fp_fn_breakdown.png   — Bar + pie chart of TP/FP/FN/TN
  fig5_gt_vs_prediction.png     — Ground truth vs model predictions
